##### NanoMaker use / execution workflow
This notebook will allow you to generate protein binding pockets for any chemical provided in SMILES format. The comments in each of the cells will walk you through.

In [1]:
from paths import *
from nano_maker.utility import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import *

In [2]:
# initialize NanoMaker system

model = NanoMaker(
    version_code=version_code,
    skeleton_weight_filename=skeleton_weight_filename,
    naanobot_weight_filename=naanobot_weight_filename,
    skeleton_cfg=skeleton_config,
    naanobot_cfg=naanobot_config,
    radial_cfg=radial_config)

In [3]:
# pt 1 -> let's create a binding pocket for a given drug
# - generate the binding pocket data
# - then save it to a .nanopkt file format
drug_i_want_to_deliver = "[H][C@]12COCCN1CCN(C[C@@H]1N3CC[C@H]3CN3C[C@@]4(CCCc5cc(Cl)ccc45)COc4ccc(cc34)C(=O)NS(=O)(=O)[C@H](C)[C@@H](C)C\C=C1/F)C2 |r,t:55|"
model.ingest_chemical(drug_i_want_to_deliver)

# you won't always need to save it to a file format but it's handy.

Chemical Ingested: [H][C@]12COCCN1CCN(C[C@@H]1N3CC[C@H]3CN3C[C@@]4(CCCc5cc(Cl)ccc45)COc4ccc(cc34)C(=O)NS(=O)(=O)[C@H](C)[C@@H](C)C\C=C1/F)C2 |r,t:55|
Scaffold: O=C1NS(=O)(=O)CCC/C=C/[C@H](CN2CCN3CCOC[C@@H]3C2)N2CC[C@H]2CN2C[C@@]3(CCCc4ccccc43)COc3ccc1cc32


<>:4: SyntaxWarning: invalid escape sequence '\C'
<>:4: SyntaxWarning: invalid escape sequence '\C'
/tmp/ipykernel_200154/2655202711.py:4: SyntaxWarning: invalid escape sequence '\C'
  drug_i_want_to_deliver = "[H][C@]12COCCN1CCN(C[C@@H]1N3CC[C@H]3CN3C[C@@]4(CCCc5cc(Cl)ccc45)COc4ccc(cc34)C(=O)NS(=O)(=O)[C@H](C)[C@@H](C)C\C=C1/F)C2 |r,t:55|"


In [4]:
pocket_data = model.generate_nanopkt_data(sampling_temp=0.3)
# sampling temperature is on a 0-1 float scale -> determines amino acid sampling strictness
# lower <= 0.2 -> fully optimized pockets but not realistic
# balanced 0.2 < 'x' <= 0.5  ->  balance between biochemical optimization and diversity
# higher 0.5 < 'x' <= 1 -> flatter prob. distribution, exploration-oriented
# essentially random: 1+
print(type(pocket_data))

<class 'dict'>


In [5]:
save_nano_pocket(pocket_data=pocket_data, filename="test_drug")

Pocket data for [H][C@]12COCCN1CCN(C[C@@H]1N3CC[C@H]3CN3C[C@@]4(CCCc5cc(Cl)ccc45)COc4ccc(cc34)C(=O)NS(=O)(=O)[C@H](C)[C@@H](C)C\C=C1/F)C2 |r,t:55| saved:
Find nano pocket file in: <_io.TextIOWrapper name='/home/elliot/Desktop/LOCAL_PROJECTS/nano_maker/src/output_container/test_drug.nanopkt' mode='w' encoding='UTF-8'>


True

In [6]:
nanopkt_data = load_nano_pocket(filename="test_drug")
if nanopkt_data == pocket_data:
    print("Data Aligned")
# ensure the file saving / loading workflow is consistent in loading the data

Data Aligned


Now that we have generated a "nanopocket" for a given smiles and its data is saved, we can visualize the generated protein binding pocket with the following cell / module.

In [7]:
from nano_maker.pocketwatcher import PocketWatcher

visualizer = PocketWatcher(naanoeng_config=naanoeng_config, pocket_config=pocketwatcher_config)
visualizer.ingest_file(pocket_filepath="test_drug")
visualizer.visualize_pocket(identifier="polar_character")
# the identifier parameter allows users to distinguish the pocket's amino acids from each other either by plain cosmetic or by their biochemical properties. These are actually "summaries" of many features from the feature vectors engineered for NAAnoBot.

# purely cosmetic identifier settings
# [1] 'skeleton' -> just shows amino acids as white
# [2] 'color_code' -> color codes amino acids
# biochemical property identifier settings
# [3] 'polar_character' -> summary metric of AA's electrical environment, greater = polar
# [4] 'hydrophobicity' -> greater = hydrophobic, lower = hydrophilic
# [5] 'flexibility' -> composite metric of AA's flexibility, greater = more freedom, lower = restricted
# [6] 'steric_accessibility' -> AA's size, weight, and exposed surface, greater = larger / more exposed

PocketWatcher also allows one to generate reports on the generated pockets for detailed quantitative analyses on the produced pockets.

In [8]:
print(visualizer.pocket_report())

BINDING POCKET REPORT
Sampling temperature: 0.3
Amino acid sequence:VSTRAEDRDQHWISGMTFFYQHSGNPEEVYDMTAKCGPFIMFGAQLGSWSKKRSEFSFGATAECL
Raw Statistics

Section 1: Biochemical Property Summary
|-- average_polar_character: 0.205
|-- polar_character_deviation_pct: 11.069
|-- average_hydrophobicity: 0.174
|-- hydrophobicity_deviation_pct: -24.712
|-- average_flexibility: 3.578
|-- flexibility_deviation_pct: 10.356
|-- average_steric: 2.236
|-- steric_deviation_pct: -2.917

Section 2: Geometric Analysis
|-- X range: 21.494
|-- Y range: 24.127
|-- Z range: 12.341
===[ Notable Binding Pocket Characteristics ]===
slightly polar_character, moderately hydrophilic, slightly flexible, chamber-style binding pocket
